# NB04 — R-Squared: What Does It Actually Measure?

> **StatQuest: "R-squared is just the square of the correlation... but let's understand WHY."**

---

## The main ideas are:

1. Without a model, your best prediction for any y is just the **mean** (y_bar)
2. The regression model improves on that — it explains SOME of the variance
3. R^2 measures exactly HOW MUCH of the variance the model explains
4. R^2 = 1 means perfect. R^2 = 0 means your line is no better than guessing the mean.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

def flow_diagram(steps, title, colors=None, notes=None, figsize=(14, 2.8)):
    """Draw a horizontal flow diagram.  steps = list of strings."""
    n = len(steps)
    default_colors = [
        '#1565C0','#2E7D32','#E65100','#6A1B9A',
        '#00695C','#AD1457','#37474F','#4E342E',
        '#0277BD','#558B2F',
    ]
    colors = (colors or default_colors[:n]) + default_colors
    notes  = notes or [''] * n

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(-0.3, n * 3.1)
    ax.set_ylim(-1.2, 2.4)
    ax.axis('off')

    bw, bh = 2.6, 1.3
    for i, (step, color, note) in enumerate(zip(steps, colors, notes)):
        x = i * 3.1
        box = FancyBboxPatch((x, 0.2), bw, bh,
                             boxstyle="round,pad=0.12",
                             facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.90)
        ax.add_patch(box)
        ax.text(x + bw/2, 0.2 + bh/2, step,
                ha='center', va='center', fontsize=8.5,
                color='white', fontweight='bold', multialignment='center')
        if note:
            ax.text(x + bw/2, 0.02, note,
                    ha='center', va='top', fontsize=7, color='#555', style='italic')
        if i < n - 1:
            ax.annotate('',
                xy=(x + bw + 0.38, 0.2 + bh/2),
                xytext=(x + bw + 0.08, 0.2 + bh/2),
                arrowprops=dict(arrowstyle='->', color='#444', lw=2.2))

    ax.set_title(title, fontsize=11, fontweight='bold', pad=6, color='#222')
    plt.tight_layout(pad=0.4)
    plt.show()

flow_diagram(
    steps=[
        'Baseline model:\ny_hat = y_bar\n(just the mean)',
        'Compute SS_tot:\nsum(y - y_bar)^2\ntotal variance in y',
        'Fit regression\nmodel:\ny_hat = b0 + b1*x',
        'Compute SS_res:\nsum(y - y_hat)^2\nunexplained variance',
        'SS_reg =\nSS_tot - SS_res\nexplained variance',
        'R^2 =\nSS_reg / SS_tot\n= 1 - SS_res/SS_tot',
        'Interpretation:\n0 = no improvement\n1 = perfect fit',
    ],
    title='NB04 Conceptual Flow: R-Squared via Sum-of-Squares Decomposition',
    colors=['#37474F','#1565C0','#2E7D32','#E65100','#6A1B9A','#C62828','#00695C'],
    figsize=(16, 2.8),
)


## The key decomposition

```
SS_tot = SS_reg + SS_res        (always true — variance partitions perfectly)

SS_tot = sum(y_i - y_bar)^2    <- total spread in y  (baseline model)
SS_res = sum(y_i - y_hat_i)^2  <- unexplained spread  (residuals)
SS_reg = sum(y_hat_i - y_bar)^2 <- explained spread   (what regression adds)

R^2 = 1 - SS_res / SS_tot   =   SS_reg / SS_tot
```

**StatQuest framing:** imagine SS_tot is $100. If SS_res = $20, then SS_reg = $80, so R^2 = 0.80. The model explained 80% of the variation in y.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

np.random.seed(42)
X = np.linspace(1, 10, 40)
y = 3*X + np.random.normal(0, 4, 40) + 5

m     = LinearRegression().fit(X.reshape(-1,1), y)
y_hat = m.predict(X.reshape(-1,1))
y_bar = y.mean()

ss_tot = np.sum((y - y_bar)**2)
ss_res = np.sum((y - y_hat)**2)
ss_reg = np.sum((y_hat - y_bar)**2)
r2     = 1 - ss_res/ss_tot

print("SS decomposition")
print(f"  SS_tot = {ss_tot:.2f}   (total variance in y)")
print(f"  SS_res = {ss_res:.2f}   (unexplained — residuals)")
print(f"  SS_reg = {ss_reg:.2f}   (explained by regression)")
print(f"  SS_tot - SS_reg - SS_res = {ss_tot-ss_reg-ss_res:.10f}  (exactly 0)")
print(f"  R^2    = 1 - {ss_res:.2f}/{ss_tot:.2f} = {r2:.4f}")
print(f"  Meaning: the model explains {r2*100:.1f}% of the variance in y")


In [ ]:
# Visualise all three SS components on the same plot
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
configs = [
    (y_bar*np.ones_like(X), 'SS_tot',  y_bar,  '#1565C0', 'Each bar = (y_i - y_bar)^2\n= Total variance in y'),
    (y_hat,                  'SS_res',  y_hat,  '#C62828', 'Each bar = (y_i - y_hat_i)^2\n= Unexplained variance'),
    (y_hat,                  'SS_reg',  y_bar,  '#2E7D32', 'Each bar = (y_hat_i - y_bar)^2\n= Explained variance'),
]

for ax, (ref_line, ss_name, lower, color, desc) in zip(axes, configs):
    ax.scatter(X, y, s=20, color='steelblue', alpha=0.7, zorder=3)
    ax.plot(X, y_hat, 'r-', linewidth=1.5, label='Fitted line')
    ax.axhline(y_bar, color='gray', linestyle='--', linewidth=1.2, label='y_bar')

    if ss_name == 'SS_reg':
        ax.vlines(X, y_bar, y_hat, color=color, alpha=0.6, linewidth=2)
    elif ss_name == 'SS_res':
        ax.vlines(X, y_hat, y, color=color, alpha=0.6, linewidth=2)
    else:
        ax.vlines(X, y_bar, y, color=color, alpha=0.6, linewidth=2)

    ax.set_title(f'{ss_name}\n{desc}', fontsize=9)
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle(f'SS_tot = SS_reg + SS_res    |    R^2 = SS_reg/SS_tot = {r2:.3f}',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# Show intuitively: compare mean-only model vs regression model
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Mean-only model
ax = axes[0]
ax.scatter(X, y, s=30, color='steelblue', alpha=0.7, zorder=3)
ax.axhline(y_bar, color='gray', linestyle='--', linewidth=2.5, label=f'Mean model: y = {y_bar:.1f}')
ax.vlines(X, y_bar, y, color='#1565C0', alpha=0.5, linewidth=1.5)
ax.set_title(f'Baseline: always predict mean\nSS_tot = {ss_tot:.1f}  (R^2 = 0.0)', fontsize=10)
ax.legend(); ax.grid(alpha=0.3)

# Regression model
ax = axes[1]
ax.scatter(X, y, s=30, color='steelblue', alpha=0.7, zorder=3)
ax.plot(X, y_hat, 'r-', linewidth=2.5, label='Regression line')
ax.vlines(X, y_hat, y, color='#C62828', alpha=0.5, linewidth=1.5)
ax.set_title(f'Regression: predict with x\nSS_res = {ss_res:.1f}  (R^2 = {r2:.3f})', fontsize=10)
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('R^2 answers: "How much better is the regression line than just guessing the mean?"',
             fontsize=11)
plt.tight_layout(); plt.show()


## What R^2 does NOT tell you

| Misconception | Truth |
|---------------|-------|
| High R^2 -> model is correct | Wrong relationship can still have high R^2 |
| R^2 = 0.9 means the model is good | Depends on the domain (astronomy vs. social science) |
| Adding predictors improves the model | R^2 always increases — use Adjusted R^2 |
| R^2 measures prediction accuracy | It measures variance explained, not forecast accuracy |

**Adjusted R^2 corrects for number of predictors:**
```
R^2_adj = 1 - (1-R^2) * (n-1) / (n-k-1)
```


In [ ]:
# Demonstrate Adjusted R^2 vs R^2
import numpy as np
from sklearn.linear_model import LinearRegression

np.random.seed(0)
n = 60
y_rand = np.random.randn(n)

r2s, r2adjs = [], []
for k in range(1, 25):
    X_noise = np.random.randn(n, k)
    m = LinearRegression().fit(X_noise, y_rand)
    yh = m.predict(X_noise)
    ssr = np.sum((y_rand-yh)**2)
    sst = np.sum((y_rand-y_rand.mean())**2)
    r2  = 1 - ssr/sst
    r2a = 1 - (1-r2)*(n-1)/(n-k-1)
    r2s.append(r2); r2adjs.append(r2a)

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(range(1,25), r2s,    'o-', label='R^2',          color='crimson',   linewidth=2)
plt.plot(range(1,25), r2adjs, 's--',label='Adjusted R^2', color='steelblue', linewidth=2)
plt.axhline(0, color='gray', linewidth=0.8)
plt.xlabel('Number of NOISE (useless) predictors'); plt.ylabel('R^2 score')
plt.title('R^2 rises with irrelevant features; Adjusted R^2 correctly stays near 0')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


## Key Takeaways

| Metric | Formula | Meaning |
|--------|---------|---------|
| SS_tot | sum(y - y_bar)^2 | Total variability in y |
| SS_res | sum(y - y_hat)^2 | What the model COULDN'T explain |
| SS_reg | SS_tot - SS_res | What the model DID explain |
| R^2 | 1 - SS_res/SS_tot | Fraction of variance explained (0 to 1) |
| Adjusted R^2 | 1 - (1-R^2)(n-1)/(n-k-1) | R^2 penalised for adding useless predictors |

**Next: NB05 — how we decide whether a coefficient is statistically significant.**
